# **CVL Final Project: Traffic Density Estimation**

by **Group 1**:

- Daffa Aryza Pasya (24/532884/PA/22549)
- Hazelleno Ian Purnomo (24/546598/PA/23206)
- Matthew Harry Indriadi (24/532723/PA/22527)
- Rafaela Anabel Purba (24/547080/PA/23218)
- Ryan Ethan Halim (24/536718/PA/22765)

Under the supervision of **Wahyono, S. Kom., Ph.D.**

This notebook implements the traditional computer vision pipeline proposed for traffic density estimation with the following pipeline.

1. **Grayscale Conversion**: luminosity method
3. **Gaussian Blur**: noise suppression
4. **Background Subtraction**: foreground (vehicle) detection via mean background model
5. **Binary Segmentation (Thresholding)**: Otsu's method
6. **Density Calculation**: pixel-ratio analysis

Evaluation is performed using a **pixel-level confusion matrix** (treating each pixel as a sample), producing accuracy, precision, recall, and F1 score against the COCO segmentation ground truth.

## Dependencies

This assignment primarily uses [numpy](https://numpy.org) for array computations, [OpenCV](https://opencv.org) for image loading and processing functions, [matplotlib](https://matplotlib.org) and [sklearn](https://scikit-learn.org) for visualizations, as well as [pycocotools](https://pypi.org/project/pycocotools) for loading the annotated COCO datasets.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

from pathlib import Path
from pycocotools.coco import COCO
from pycocotools import mask as coco_mask_util
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Suppresses pycocotools stdout
import io, contextlib

## Loading Assets

We load the COCO segmentation annotations and corresponding images. The ground-truth segmentation masks (binary, per-pixel) will be used later for evaluation. The images were annotated externally using [Roboflow](https://roboflow.com).

In [ ]:
ANN_PATH = Path("dataset/train/_annotations.coco.json")
IMAGE_DIR = Path("dataset/train/")

with contextlib.redirect_stdout(io.StringIO()):
    coco = COCO(ANN_PATH)

# Gathers image IDs that have annotations
img_ids = sorted(coco.getImgIds())[:200]
print(f"Total images in dataset : {len(img_ids)}")

ann_ids = coco.getAnnIds(imgIds=img_ids)
print(f"Total annotations       : {len(ann_ids)}")

## Auxiliary Functions

These are functions implemented to be reused in visualizations throughout the notebook.

In [ ]:
# Loads an image as float32 RGB [0, 1]
def load_image(coco_img_info) -> np.ndarray:
    path = IMAGE_DIR / coco_img_info['file_name']
    img_bgr = cv2.imread(path)
    assert img_bgr is not None, f"Could not read {path}"
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    return img_rgb.astype(np.float32) / 255.0


# Return a binary (bool) segmentation mask for all annotated vehicles in an image
def get_gt_mask(coco: COCO, img_id: int, height: int, width: int) -> np.ndarray:
    ann_ids = coco.getAnnIds(imgIds=[img_id])
    mask = np.zeros((height, width), dtype=bool)
    for ann in coco.loadAnns(ann_ids):
        rle = coco.annToRLE(ann)  # type: ignore
        m   = coco_mask_util.decode(rle).astype(bool)
        mask |= m

    return mask


def show_pair(img: np.ndarray, mask: np.ndarray, title_left="Image", title_right="Mask"):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(img)
    axes[0].set_title(title_left, fontsize=13)
    axes[0].axis('off')
    axes[1].imshow(mask, cmap='gray')
    axes[1].set_title(title_right, fontsize=13)
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

## Dataset Preview

Below, we visualize a sample image alongside its ground-truth segmentation mask, as to confirm the annotations are loaded correctly.

In [ ]:
sample_id   = img_ids[0]
sample_info = coco.loadImgs([sample_id])[0]
sample_img  = load_image(sample_info)
sample_gt   = get_gt_mask(coco, sample_id, sample_info['height'], sample_info['width'])

show_pair(sample_img, sample_gt, "Original frame", "Ground-truth vehicle mask")
print(f"Image size : {sample_info['width']} x {sample_info['height']}")

## Pipeline

### Grayscale Conversion

Color information is discarded using the ITU-R BT.709 luminosity formula:

$$Y = 0.2126 \cdot R + 0.7152 \cdot G + 0.0722 \cdot B$$

It is implemented in the function below.

In [ ]:
# Converts an RGB float32 image to a single-channel grayscale image using the luminosity method
def to_grayscale(image: np.ndarray) -> np.ndarray:
    # ITU-R BT.709 coefficients
    return (0.2126 * image[..., 0]
          + 0.7152 * image[..., 1]
          + 0.0722 * image[..., 2]).astype(np.float32)

Below is the application for demonstration.

In [ ]:
gray_sample = to_grayscale(sample_img)
show_pair(sample_img, gray_sample, "Original (color)", "Grayscale")

### Noise Removal

Compression artefacts and sensor noise are suppressed by convolving the image with a Gaussian kernel:

$$G(x, y) = \frac{1}{2\pi\sigma^2} \exp\!\left(-\frac{x^2+y^2}{2\sigma^2}\right)$$

It is indirectly implemented in the function below.

In [ ]:
# Applies Gaussian blur to a single-channel float32 image
def gaussian_blur(image: np.ndarray, kernel_size: int = 5, sigma: float = 1.0) -> np.ndarray:
    # `kernel_size` must be odd
    ksize = kernel_size if kernel_size % 2 == 1 else kernel_size + 1
    blurred = cv2.GaussianBlur(image, (ksize, ksize), sigmaX=sigma, sigmaY=sigma)
    return blurred.astype(np.float32)

Below is the application for demonstration.

In [ ]:
blurred_sample = gaussian_blur(gray_sample, kernel_size=5, sigma=1.0)
show_pair(gray_sample, blurred_sample, "Grayscale", "After Gaussian blur")

### Background Subtraction

A background model is built by computing the per-pixel mean across all available frames. Since the camera is fixed, the temporal mean of a traffic scene converges toward the empty road.

$$B(x, y) = \frac{1}{N}\sum_{i=1}^{N} F_i(x, y)$$

Each frame is then subtracted from the model:

$$\hat{S}(x,y) = |F(x,y) - B(x,y)|$$

Both of these operations are respectively implemented in the functions below.

In [ ]:
# Compute the per-pixel temporal mean of a list of single-channel float32 frames;
# this serves as the empty-road background model.
def build_background_model(images: list[np.ndarray]) -> np.ndarray:
    stack = np.stack(images, axis=0)  # N × H × W
    return stack.mean(axis=0).astype(np.float32)


# Returns the absolute difference between a frame and the background model
def subtract_background(frame: np.ndarray, background: np.ndarray) -> np.ndarray:
    return np.abs(frame - background).astype(np.float32)

Below is the application for demonstration. The entire dataset must be processed through the preprocessing steps before to reconstruct the background model.

In [ ]:
print("Loading and preprocessing all frames to build background model...")

all_gray_frames: list[np.ndarray] = []
coco_images_info = []

for img_id in img_ids:
    # Does the previous steps to all images
    info = coco.loadImgs([img_id])[0]
    img = load_image(info)
    gray = to_grayscale(img)
    blurred = gaussian_blur(gray)
    all_gray_frames.append(blurred)
    coco_images_info.append(info)

background_model = build_background_model(all_gray_frames)
print(f"Background model built from {len(all_gray_frames)} frame(s).")

plt.figure(figsize=(6, 5))
plt.imshow(background_model, cmap='gray')
plt.title("Background model (mean frame)")
plt.axis('off')
plt.tight_layout()
plt.show()

They are then subtracted to obtain the foreground.

In [ ]:
diff_sample = subtract_background(all_gray_frames[0], background_model)
show_pair(all_gray_frames[0], diff_sample, "Preprocessed frame", "Background subtraction result")

### Binary Segmentation

The difference image is converted to a binary mask using Otsu's method, which automatically selects the optimal threshold $T^*$ by minimising intra-class variance:

$$\text{IsForeground}(x,y) \equiv \hat{S}(x,y) > T^*$$

It is indirectly implemented in the function below.

In [ ]:
# Applies Otsu's global threshold to a float32 single-channel difference image
def otsu_threshold(diff_image: np.ndarray) -> np.ndarray:
    # cv2.threshold expects uint8
    uint8 = (diff_image * 255).clip(0, 255).astype(np.uint8)
    _, binary = cv2.threshold(uint8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return binary > 0

Below is the application for demonstration.

In [ ]:
binary_sample = otsu_threshold(diff_sample)
show_pair(diff_sample, binary_sample, "Difference image", "Binary mask (Otsu)")

### Density Calculation

Traffic density is the fraction of foreground pixels (detected vehicles) within the ROI:

$$\text{Density} = \frac{\sum \text{Object Pixels}}{\text{Total ROI Pixels}} \times 100\%$$

A higher value indicates denser traffic.

In [ ]:
# Returns traffic density as a percentage [0, 100]
def compute_density(binary_mask: np.ndarray) -> float:
    return float(binary_mask.sum()) / binary_mask.size * 100.0

Below is the application for demonstration.

In [ ]:
density_sample = compute_density(binary_sample)
print(f"Traffic density for sample frame : {density_sample:.2f}%")

## Full Pipeline

Every step is effectively done throughout all frames in the dataset.

In [ ]:
predicted_masks: list[np.ndarray] = []  # Binary H×W bool
gt_masks: list[np.ndarray] = []  # Binary H×W bool
density_scores: list[float] = []
gt_density_scores: list[float] = []

print("Running pipeline on all frames...")

for i, (img_id, info, gray_blurred) in enumerate(
        zip(img_ids, coco_images_info, all_gray_frames)):
    h, w = info['height'], info['width']

    # Background subtraction + Otsu thresholding
    diff = subtract_background(gray_blurred, background_model)
    pred = otsu_threshold(diff)

    # Ground-truth mask
    gt = np.zeros((h, w), dtype=bool)
    ann_ids_frame = coco.getAnnIds(imgIds=[img_id])
    for ann in coco.loadAnns(ann_ids_frame):
        try:
            rle = coco.annToRLE(ann)  # type: ignore
            m   = coco_mask_util.decode(rle).astype(bool)
            gt |= m
        except Exception as e:
            print(f"Warning: Skipping annotation ID {ann.get('id','?')} — {e}")

    predicted_masks.append(pred)
    gt_masks.append(gt)
    density_scores.append(compute_density(pred))
    gt_density_scores.append(compute_density(gt))

    if (i + 1) % max(1, len(img_ids) // 5) == 0 or i == len(img_ids) - 1:
        print(f"  Processed {i+1}/{len(img_ids)} frames")

print("Done.")

## Evaluation

### Qualitative Demonstration

Below, we overlay the predicted mask and ground-truth mask on a few sample frames.

In [ ]:
# Returns an RGB image with prediction (red) and GT (green) overlaid
def overlay_masks(img: np.ndarray, pred: np.ndarray, gt: np.ndarray) -> np.ndarray:
    vis = img.copy()
    vis[pred & ~gt]  = [1.0, 0.2, 0.2]   # FP  → red
    vis[gt  & ~pred] = [0.2, 1.0, 0.2]   # FN  → green
    vis[pred & gt]   = [1.0, 1.0, 0.0]   # TP  → yellow
    return vis

In [ ]:
n_show = min(4, len(img_ids))
fig, axes = plt.subplots(n_show, 3, figsize=(15, 4 * n_show))

if n_show == 1:
    axes = [axes]

legend_patches = [
    mpatches.Patch(color=(1,1,0),   label='TP (pred ∩ gt)'),
    mpatches.Patch(color=(1,0.2,0.2), label='FP (pred \\ gt)'),
    mpatches.Patch(color=(0.2,1,0.2), label='FN (gt \\ pred)'),
]

for row, idx in enumerate(range(0, len(img_ids), max(1, len(img_ids) // n_show))[:n_show]):
    info  = coco_images_info[idx]
    img   = load_image(info)
    pred  = predicted_masks[idx]
    gt    = gt_masks[idx]
    dens  = density_scores[idx]

    axes[row][0].imshow(img)
    axes[row][0].set_title(f"Frame {idx+1}: {info['file_name']}", fontsize=10)
    axes[row][0].axis('off')

    axes[row][1].imshow(pred, cmap='gray')
    axes[row][1].set_title(f"Predicted mask  (density = {dens:.1f}%)", fontsize=10)
    axes[row][1].axis('off')

    axes[row][2].imshow(overlay_masks(img, pred, gt))
    axes[row][2].set_title("Overlay (TP=yellow, FP=red, FN=green)", fontsize=10)
    axes[row][2].axis('off')

    if row == 0:
        axes[row][2].legend(handles=legend_patches, loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()

### Confusion Matrix

Each pixel's prediction is treated as a binary determiner. Thus, a global confusion matrix is built by flattening and concatenating all predicted and ground-truth masks, as visualized below.

|                     | Pred. Positive | Pred. Negative |
| ------------------: | :------------: | :------------: |
| **Actual Positive** |       TP       |       FN       |
| **Actual Negative** |       FP       |       TN       |

In [ ]:
y_true = np.concatenate([gt.ravel().astype(int) for gt in gt_masks])
y_pred = np.concatenate([pm.ravel().astype(int) for pm in predicted_masks])

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=["Background", "Vehicle"])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, colorbar=True, cmap='Blues', values_format=',d')
ax.set_title("Pixel-level Confusion Matrix", fontsize=13)
plt.tight_layout()
plt.show()

From these, the following are computed as heuristics:

$$
\text{Accuracy}  = \frac{TP + TN}{TP + TN + FP + FN}
\qquad
\text{Precision} = \frac{TP}{TP + FP}
\qquad
\text{Recall}    = \frac{TP}{TP + FN}
\qquad
\text{F1}        = \frac{2 \cdot TP}{2 \cdot TP + FP + FN}
$$

In [ ]:
total  = tn + fp + fn + tp
acc    = (tp + tn) / total
prec   = tp / (tp + fp) if (tp + fp) > 0 else 0.0
rec    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1     = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0.0

print("=" * 48)
print(f"  Accuracy       : {acc*100:.4f}%")
print(f"  Precision      : {prec*100:.4f}%")
print(f"  Recall         : {rec*100:.4f}%")
print(f"  F1 Score       : {f1*100:.4f}%")
print("=" * 48)

## Prediction vs. Truth across Frames

The chart below shows the estimated traffic density (%) for every image in the dataset, giving an at-a-glance summary of congestion levels throughout the sequence.

In [ ]:
frames_x = range(1, len(density_scores) + 1)

plt.figure(figsize=(max(8, len(density_scores) // 2), 5))
plt.plot(frames_x, density_scores,    marker='o', linewidth=1.5, markersize=4,
         label='Predicted density',    color='steelblue')
plt.plot(frames_x, gt_density_scores, marker='s', linewidth=1.5, markersize=4,
         label='Ground-truth density', color='tomato', linestyle='--')
plt.xlabel("Frame index")
plt.ylabel("Traffic density (%)")
plt.title("Estimated vs Ground-Truth Traffic Density per Frame")
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 48)
print(f"{'':20s} {'Predicted':>12s}  {'Ground Truth':>12s}")
print(f"{'Mean density':<20s} {np.mean(density_scores):>11.2f}%  {np.mean(gt_density_scores):>11.2f}%")
print(f"{'Median density':<20s} {np.median(density_scores):>11.2f}%  {np.median(gt_density_scores):>11.2f}%")
print(f"{'Max density':<20s} {np.max(density_scores):>11.2f}%  {np.max(gt_density_scores):>11.2f}%")
print(f"{'Min density':<20s} {np.min(density_scores):>11.2f}%  {np.min(gt_density_scores):>11.2f}%")
print("=" * 48)